In [ ]:
import os
import json
import csv
from bert_score import score

root_dir = "../../../yuxuan-anomal/results"
save_dir = os.path.join(root_dir, "bertscore")

# 创建保存文件夹
os.makedirs(save_dir, exist_ok=True)

folder_results = {}

# 遍历每个模型文件夹
for folder in sorted(os.listdir(root_dir)):
    folder_path = os.path.join(root_dir, folder)
    if not os.path.isdir(folder_path):
        continue

    # 跳过我们新建的保存目录
    if folder == "bertscore":
        continue

    preds = []
    refs = []

    # 遍历 three json files: good, logical, structure
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            json_path = os.path.join(folder_path, filename)
            print("Loading:", json_path)

            with open(json_path, "r") as f:
                data = json.load(f)

            for item in data:
                preds.append(item["predicted_answer"])
                refs.append(item["true_anwser"])  # 你的字段名

    # 如果为空就跳过
    if len(preds) == 0:
        continue

    # 计算 BERTScore
    P, R, F1 = score(preds, refs, lang="en", model_type="roberta-large")

    precision = P.mean().item()
    recall = R.mean().item()
    f1 = F1.mean().item()

    # 保存到字典
    folder_results[folder] = {
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

    # --- 保存单独 JSON 文件 ---
    save_path = os.path.join(save_dir, f"{folder}.json")
    with open(save_path, "w") as f:
        json.dump(folder_results[folder], f, indent=4)
    print(f"Saved: {save_path}")

# --- 保存汇总 CSV ---
csv_path = os.path.join(save_dir, "all_results.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Folder", "Precision", "Recall", "F1"])

    for folder, metrics in folder_results.items():
        writer.writerow([folder, metrics["Precision"], metrics["Recall"], metrics["F1"]])

print(f"\nAll results saved to {csv_path}")
